# Benchmark — pgvector (PostgreSQL 18)
## Vade Mecum EC134/2024

Pipeline completo de ingestão e busca com o backend **PostgreSQL 18 + pgvector 0.8.2**.

| Passo | Descrição |
|-------|-----------|
| 0 | Verificar conexão ao PostgreSQL |
| 1 | Extrair texto do PDF (PyMuPDF) |
| 2 | Chunking jurídico por artigo |
| 3 | Gerar embeddings (multilingual-e5-large, 1024 dims) |
| 4 | Indexar no pgvector (tabela + HNSW cosine) |
| 5 | Buscas de teste — latência por query |
| 6 | Estatísticas de storage (pg_total_relation_size) |

### Pré-requisitos

```bash
# Suba o PostgreSQL + pgvector
docker compose -f docker-compose.postgres.yml up postgres -d

# Instale as dependências Python
uv sync --group postgres
```

> Modelo `intfloat/multilingual-e5-large` (~2 GB) baixado automaticamente na 1ª execução.  
> `RECRIAR = True` → recria a tabela e reindexa; `False` → reutiliza dados já indexados.


In [6]:
import sys
import statistics
import time
from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CAMINHO_PDF = ROOT / 'data' / 'Vade_mecum_EC134_2024.pdf'
RECRIAR = True  # False = reutiliza dados já indexados (pula ingestão)

QUERIES = [
    'princípios do tratamento de dados pessoais LGPD',
    'direitos fundamentais habeas corpus mandado de segurança',
    'rescisão do contrato de trabalho aviso prévio CLT',
    'direitos do consumidor código de defesa vício do produto',
    'imposto de renda pessoa física alíquota tributação',
    'usucapião direito de propriedade prazo posse',
    'crime de furto pena reclusão detenção código penal',
    'ação civil pública interesse difuso ministério público',
    'licitação contrato administrativo dispensa inexigibilidade',
    'criança adolescente ECA medida socioeducativa',
]

_t = {}  # acumula tempos para o resumo final
assert CAMINHO_PDF.exists(), f'PDF não encontrado: {CAMINHO_PDF}'
print(f'✔  {CAMINHO_PDF.name}  ({CAMINHO_PDF.stat().st_size / 1e6:.1f} MB)')

✔  Vade_mecum_EC134_2024.pdf  (24.4 MB)


## Passo 0 — Conectividade

Verifica se o PostgreSQL está rodando na porta 5432 e se a extensão `vector` está instalada.

In [7]:
from ana.storage.pgvector_store import IndexadorPgVector

idx = IndexadorPgVector()
if not idx.verificar_conexao():
    raise RuntimeError(
        'PostgreSQL indisponível.\n'
        'Execute: docker compose -f docker-compose.postgres.yml up postgres -d\n'
        'E:       uv sync --group postgres'
    )
print('✔  PostgreSQL + pgvector acessível')
print(f'   Tabelas: {idx.listar_colecoes()}')

✔  PostgreSQL + pgvector acessível
   Tabelas: []


## Passo 1 — Extração de texto do PDF

Usa **PyMuPDF** para extrair o texto bruto. O Vade Mecum tem ~4.4 M de caracteres.

In [8]:
from ana.rag.ingestao import extrair_texto_pdf

t0 = time.perf_counter()
texto = extrair_texto_pdf(CAMINHO_PDF)
t_ext = time.perf_counter() - t0
_t['extracao'] = t_ext

pdf_mb = CAMINHO_PDF.stat().st_size / 1e6
print(f'✔  {len(texto):,} caracteres extraídos  [{t_ext:.2f}s]')
print(f'   Velocidade: {pdf_mb / t_ext:.1f} MB/s')
print()
print('Primeiros 500 chars:')
print(texto[:500])

2026-02-26 14:04:13.920 | INFO     | ana.rag.ingestao:extrair_texto_pdf:233 - PDF extraído: Vade_mecum_EC134_2024.pdf (4393818 chars)


✔  4,393,818 caracteres extraídos  [1.26s]
   Velocidade: 19.4 MB/s

Primeiros 500 chars:



Brasília – DF
VADE 
MECUM

Mesa Diretora do Senado Federal
Biênio 2023–2024
Senador Rodrigo Pacheco
PRESIDENTE
Senador Veneziano Vital do Rêgo
PRIMEIRO-VICE-PRESIDENTE
Senador Rodrigo Cunha
SEGUNDO-VICE-PRESIDENTE
Senador Rogério Carvalho
PRIMEIRO-SECRETÁRIO
Senador Weverton
SEGUNDO-SECRETÁRIO
Senador Chico Rodrigues
TERCEIRO-SECRETÁRIO
Senador Styvenson Valentim
QUARTO-SECRETÁRIO
SUPLENTES DE SECRETÁRIO
1ª suplente: Senadora Mara Gabrilli
2ª suplente: Senadora Ivete da Silveira
3o suplente: 


## Passo 2 — Chunking jurídico por artigo

O texto é dividido em chunks usando o padrão `Art. N`, preservando o texto completo de cada dispositivo legal.

In [9]:
from ana.rag.ingestao import processar_documento
from ana.rag.modelos import TipoDocumento, VigenciaStatus

t0 = time.perf_counter()
chunks = processar_documento(
    texto=texto,
    fonte='Vade Mecum EC134/2024',
    tipo=TipoDocumento.LEI_FEDERAL,
    vigencia=VigenciaStatus.ATIVA,
)
t_chk = time.perf_counter() - t0
_t['chunking'] = t_chk

tamanhos = [len(c.texto) for c in chunks]
print(f'✔  {len(chunks):,} chunks gerados  [{t_chk:.2f}s]')
print(f'   média={statistics.mean(tamanhos):.0f}  min={min(tamanhos)}  max={max(tamanhos)}  mediana={statistics.median(tamanhos):.0f} chars')

2026-02-26 14:07:16.135 | INFO     | ana.rag.ingestao:chunkar_texto_juridico:191 - Chunking concluído: 7889 artigos extraídos de 'Vade Mecum EC134/2024'


✔  7,889 chunks gerados  [182.21s]
   média=539  min=17  max=22664  mediana=305 chars


In [10]:
# Amostra dos primeiros 10 chunks
pd.DataFrame([
    {'artigo': c.metadata.artigo, 'fonte': c.metadata.fonte, 'chars': len(c.texto), 'prévia': c.texto[:100] + '…'}
    for c in chunks[:10]
])

,artigo,fonte,chars,prévia
0,Art. 1,Vade Mecum EC134/2024,505,"Art. 1o A República Federativa do \nBrasil, f..."
1,Art. 2,Vade Mecum EC134/2024,114,"Art. 2o São Poderes da União, \nindependentes..."
2,Art. 3,Vade Mecum EC134/2024,405,Art. 3o Constituem objetivos \nfundamentais ...
3,Art. 4,Vade Mecum EC134/2024,729,Art. 4o A República Federativa \ndo Brasil re...
4,Art. 5,Vade Mecum EC134/2024,14111,"Art. 5o Todos são iguais perante \na lei, sem..."
5,Art. 6,Vade Mecum EC134/2024,568,"Art. 6o São direitos sociais a \neducação, a ..."
6,Art. 7,Vade Mecum EC134/2024,4731,Art. 7o São direitos dos traba-\nlhadores urb...
7,Art. 8,Vade Mecum EC134/2024,1688,Art. 8o É livre a associação pro-\nfissional ...
8,Art. 9,Vade Mecum EC134/2024,381,"Art. 9o É assegurado o direito \nde greve, co..."
9,Art. 10,Vade Mecum EC134/2024,212,Art. 10. É assegurada a partici-\npação dos t...


## Passo 3 — Geração de embeddings

Usa **`intfloat/multilingual-e5-large`** (1024 dims) via CUDA. Na 1ª execução faz o download do modelo (~2 GB).

In [11]:
from ana.rag.embeddings import GeradorEmbeddings
from ana.config_modelos import obter_modelos

cfg_emb = obter_modelos().ativo.embeddings
print(f'Modelo  : {cfg_emb.modelo}')
print(f'Dimensão: {cfg_emb.dimensao}')
print(f'Device  : {cfg_emb.dispositivo}')

gerador = GeradorEmbeddings()
textos = [c.texto for c in chunks]

t0 = time.perf_counter()
vecs = gerador.gerar_batch(textos)
t_emb = time.perf_counter() - t0
_t['embeddings'] = t_emb

for chunk, v in zip(chunks, vecs):
    chunk.embedding = v

n = len(chunks)
print(f'\n✔  {n:,} embeddings  [{t_emb:.2f}s]  ({n / t_emb:.0f} chunks/s)')
print(f'   Shape: {len(vecs)} × {len(vecs[0])} dims')

Modelo  : intfloat/multilingual-e5-large
Dimensão: 1024
Device  : cuda


/mnt/hd/Repos/attorney-normative-assistent/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-02-26 14:07:18.579 | INFO     | ana.rag.embeddings:_carregar_modelo:73 - Carregando modelo de embeddings: intfloat/multilingual-e5-large (dispositivo=cuda)
2026-02-26 14:07:22.985 | INFO     | ana.rag.embeddings:_carregar_modelo:81 - Modelo carregado: dimensão=1024, batch_size=32
2026-02-26 14:07:22.988 | DEBUG    | ana.rag.embeddings:gerar_batch:124 - Gerando embeddings para 7889 textos...
Batches: 100%|██████████| 247/247 [01:11<00:00,  3.44it/s]



✔  7,889 embeddings  [78.48s]  (101 chunks/s)
   Shape: 7889 × 1024 dims


## Passo 4 — Indexação no pgvector

Cria (ou recria) a tabela `legislacao_brasileira` com índice HNSW (`vector_cosine_ops`) e insere os chunks em lotes de 100.

```sql
CREATE TABLE legislacao_brasileira (
    id UUID PRIMARY KEY,
    vetor vector(1024) NOT NULL,
    texto TEXT, tipo TEXT, area TEXT, vigencia TEXT, ...
    payload JSONB
);
CREATE INDEX USING hnsw (vetor vector_cosine_ops);
```

In [12]:
idx.criar_colecao_legislacao(recriar=RECRIAR)

t0 = time.perf_counter()
total = idx.indexar_chunks(chunks)
t_idx = time.perf_counter() - t0
_t['indexacao'] = t_idx

print(f'✔  {total:,} chunks indexados  [{t_idx:.2f}s]  ({total / t_idx:.0f} chunks/s)')

2026-02-26 14:08:34.990 | WARNING  | ana.storage.pgvector_store:criar_colecao:138 - Tabela pgvector apagada: legislacao_brasileira
2026-02-26 14:08:34.999 | INFO     | ana.storage.pgvector_store:criar_colecao:172 - Tabela pgvector: legislacao_brasileira (dims=1024, distância=COSINE/HNSW)
2026-02-26 14:08:35.197 | DEBUG    | ana.storage.pgvector_store:indexar_chunks:333 - pgvector: lote 1 indexado (100 chunks)
2026-02-26 14:08:35.424 | DEBUG    | ana.storage.pgvector_store:indexar_chunks:333 - pgvector: lote 2 indexado (100 chunks)
2026-02-26 14:08:35.643 | DEBUG    | ana.storage.pgvector_store:indexar_chunks:333 - pgvector: lote 3 indexado (100 chunks)
2026-02-26 14:08:35.866 | DEBUG    | ana.storage.pgvector_store:indexar_chunks:333 - pgvector: lote 4 indexado (100 chunks)
2026-02-26 14:08:36.088 | DEBUG    | ana.storage.pgvector_store:indexar_chunks:333 - pgvector: lote 5 indexado (100 chunks)
2026-02-26 14:08:36.311 | DEBUG    | ana.storage.pgvector_store:indexar_chunks:333 - pgvect

✔  7,889 chunks indexados  [20.78s]  (380 chunks/s)


## Passo 5 — Buscas de teste

10 queries jurídicas representativas. Mede apenas a latência da busca vetorial (o embedding da query não é contabilizado).

In [13]:
latencias = []
linhas = []

for query in QUERIES:
    vetor = gerador.gerar_query(query)
    t0 = time.perf_counter()
    res = idx.busca_semantica(vetor, limite=5)
    lat_ms = (time.perf_counter() - t0) * 1000
    latencias.append(lat_ms)

    melhor = res[0] if res else {}
    payload = melhor.get('payload', {})
    linhas.append({
        'query': query,
        'lat (ms)': round(lat_ms, 1),
        'artigo': payload.get('artigo', '—'),
        'score': round(melhor.get('score', 0), 3) if res else 0,
        'trecho': (payload.get('texto') or '')[:80] + '…',
    })

pd.DataFrame(linhas)

,query,lat (ms),artigo,score,trecho
0,princípios do tratamento de dados pessoais LGPD,9.9,Art. 90,0.835,Art. 90. Aplicam-se às ações \nprevistas nest...
1,direitos fundamentais habeas corpus mandado de...,8.6,Art. 656,0.865,Art. 656. Recebida a petição de \nhabeas corp...
2,rescisão do contrato de trabalho aviso prévio CLT,9.1,Art. 490,0.880,"Art. 490. O empregador que, \ndurante o prazo..."
3,direitos do consumidor código de defesa vício ...,9.0,Art. 18,0.870,Art. 18. Os fornecedores de pro-\ndutos de co...
4,imposto de renda pessoa física alíquota tribut...,8.8,Art. 99,0.863,Art. 99. Para efeito do disposto \nno inciso ...
5,usucapião direito de propriedade prazo posse,9.0,Art. 1,0.887,Art. 1.379. O exercício incontes-\ntado e con...
6,crime de furto pena reclusão detenção código p...,7.7,Art. 351,0.878,Art. 351. Promover ou facilitar a \nfuga de p...
7,ação civil pública interesse difuso ministério...,9.2,Art. 77,0.873,Art. 77. A falta de intervenção \ndo Ministér...
8,licitação contrato administrativo dispensa ine...,8.2,Art. 72,0.867,"Art. 72. O processo de contra-\ntação direta,..."
9,criança adolescente ECA medida socioeducativa,7.8,Art. 112,0.861,Art. 112. Verificada a prática de \nato infra...


In [14]:
lat_s = sorted(latencias)
p95 = lat_s[int(len(lat_s) * 0.95)]
print('Latências (ms):')
print(f'  média : {statistics.mean(latencias):.1f}')
print(f'  p50   : {statistics.median(latencias):.1f}')
print(f'  p95   : {p95:.1f}')
print(f'  max   : {max(latencias):.1f}')
_t['latencias'] = latencias

Latências (ms):
  média : 8.7
  p50   : 8.9
  p95   : 9.9
  max   : 9.9


## Passo 6 — Estatísticas de storage

Consulta o PostgreSQL com `pg_total_relation_size` para obter tamanho de dados e índices.

In [15]:
from ana.config import obter_configuracao
import psycopg

cfg = obter_configuracao()
colecao = cfg.colecao_legislacao

conn = psycopg.connect(cfg.postgres_dsn)
try:
    count = conn.execute(f'SELECT COUNT(*) FROM {colecao}').fetchone()[0]
    print(f'Linhas na tabela   : {count:,}')

    row = conn.execute('SELECT pg_size_pretty(pg_relation_size(%s))', (colecao,)).fetchone()
    print(f'Dados (tabela)     : {row[0]}')

    row = conn.execute('SELECT pg_size_pretty(pg_indexes_size(%s))', (colecao,)).fetchone()
    print(f'Índices (HNSW)     : {row[0]}')

    row = conn.execute('SELECT pg_size_pretty(pg_total_relation_size(%s))', (colecao,)).fetchone()
    print(f'Total              : {row[0]}')

    rows = conn.execute(
        'SELECT indexname, pg_size_pretty(pg_relation_size(indexname::regclass)) FROM pg_indexes WHERE tablename = %s',
        (colecao,)
    ).fetchall()
    if rows:
        print('\nDetalhamento de índices:')
        for nome_idx, tamanho in rows:
            print(f'  {nome_idx:<45} {tamanho}')
finally:
    conn.close()

Linhas na tabela   : 7,889
Dados (tabela)     : 7856 kB
Índices (HNSW)     : 62 MB
Total              : 112 MB

Detalhamento de índices:
  legislacao_brasileira_pkey                    328 kB
  legislacao_brasileira_vetor_hnsw              62 MB
  legislacao_brasileira_vigencia_idx            72 kB
  legislacao_brasileira_tipo_idx                88 kB


## Resumo Final

In [ ]:
lats  = _t['latencias']
lat_s = sorted(lats)
t_ext = _t['extracao']
t_chk = _t['chunking']
t_emb = _t['embeddings']
t_idx = _t['indexacao']
n     = len(chunks)
total_s = t_ext + t_chk + t_emb + t_idx
pdf_mb  = CAMINHO_PDF.stat().st_size / 1e6

resumo = pd.DataFrame([
    {'Etapa': 'Extração PDF',           'Tempo': f'{t_ext:.2f}s',   'Throughput': f'{pdf_mb / t_ext:.1f} MB/s'},
    {'Etapa': 'Chunking jurídico',      'Tempo': f'{t_chk:.2f}s',   'Throughput': f'{n / t_chk:.0f} chunks/s'},
    {'Etapa': 'Embeddings (e5-large)',  'Tempo': f'{t_emb:.2f}s',   'Throughput': f'{n / t_emb:.0f} chunks/s'},
    {'Etapa': 'Indexação pgvector',     'Tempo': f'{t_idx:.2f}s',   'Throughput': f'{n / t_idx:.0f} chunks/s'},
    {'Etapa': 'TOTAL ingestão',         'Tempo': f'{total_s:.1f}s', 'Throughput': ''},
    {'Etapa': 'Busca — média',          'Tempo': '',                'Throughput': f'{statistics.mean(lats):.1f} ms'},
    {'Etapa': 'Busca — p50',            'Tempo': '',                'Throughput': f'{statistics.median(lats):.1f} ms'},
    {'Etapa': 'Busca — p95',            'Tempo': '',                'Throughput': f'{lat_s[int(len(lat_s) * 0.95)]:.1f} ms'},
    {'Etapa': 'Busca — max',            'Tempo': '',                'Throughput': f'{max(lats):.1f} ms'},
])
resumo.style.set_caption('Benchmark pgvector — Vade Mecum EC134/2024')

,Etapa,Tempo,Throughput
0,Extração PDF,1.26s,19.4 MB/s
1,Chunking jurídico,182.21s,43 chunks/s
2,Embeddings (e5-large),78.48s,101 chunks/s
3,Indexação pgvector,20.78s,380 chunks/s
4,TOTAL ingestão,282.7s,
5,Busca — média,,8.7 ms
6,Busca — p50,,8.9 ms
7,Busca — p95,,9.9 ms
8,Busca — max,,9.9 ms


: 